In [11]:
# pip install yfinance pandas numpy matplotlib ta


In [12]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:.2f}".format)


In [13]:
TICKERS = [
    # Broad Market
    "SPY","QQQ","DIA","IWM",

    # Mega Cap Tech
    "AAPL","MSFT","NVDA","GOOGL","AMZN",
    "META","TSLA","AVGO","ORCL","AMD",

    # Financials
    "JPM","BAC","GS","MS","BLK",

    # Healthcare
    "LLY","UNH","JNJ","ABBV","MRK",

    # Industrials
    "CAT","GE","RTX","DE","HON",

    # Consumer
    "COST","WMT","HD","MCD","SBUX",

    # Energy
    "XOM","CVX","COP","SLB",

    # Semiconductors
    "MU","QCOM","TXN","INTC","ASML",

    # ETFs
    "VTI","VOO","SCHD","VXUS","XLK","XLF","XLE"
]
START_DATE = "2018-01-01"


In [14]:
def load_weekly_data(ticker):
    df = yf.Ticker(ticker).history(
        start=START_DATE,
        interval="1d",
        auto_adjust=True
    )

    # Keep only Close and force Series
    close = df["Close"].astype(float)

    # Weekly resample
    close = close.resample("W-FRI").last().dropna()

    return pd.DataFrame({"Close": close})


In [15]:
def compute_weekly_signals(df):
    df = df.copy()

    close = pd.Series(df["Close"].values, index=df.index)

    # Defensive checks
    assert close.ndim == 1, f"Close is not 1D: shape={close.shape}"

    df["ma40"] = close.rolling(40).mean()
    df["rsi"] = ta.momentum.RSIIndicator(close, window=14).rsi()

    df["signal"] = "HOLD"
    df.loc[(close > df["ma40"]) & (df["rsi"] < 70), "signal"] = "BUY"
    df.loc[close < df["ma40"], "signal"] = "SELL"

    return df


In [16]:
def backtest(df):
    df = df.copy()

    df["position"] = 0
    df.loc[df["signal"] == "BUY", "position"] = 1
    df.loc[df["signal"] == "SELL", "position"] = 0
    df["position"] = df["position"].ffill().fillna(0)

    df["weekly_return"] = df["Close"].pct_change()
    df["strategy_return"] = df["position"].shift(1) * df["weekly_return"]

    df["cum_market"] = (1 + df["weekly_return"]).cumprod()
    df["cum_strategy"] = (1 + df["strategy_return"]).cumprod()

    return df


In [17]:
results = {}
summaries = []

for ticker in TICKERS:
    df = load_weekly_data(ticker)
    df = compute_weekly_signals(df)
    df = backtest(df)

    results[ticker] = df

    total_return = df["cum_strategy"].iloc[-1] - 1
    market_return = df["cum_market"].iloc[-1] - 1

    summaries.append({
        "Ticker": ticker,
        "Strategy Return %": total_return * 100,
        "Market Return %": market_return * 100,
        "Outperformed": total_return > market_return
    })

summary_df = pd.DataFrame(summaries)
summary_df


,Ticker,Strategy Return %,Market Return %,Outperformed
0,SPY,126.58,210.12,False
1,QQQ,321.78,381.36,False
2,DIA,38.60,140.27,False
3,IWM,57.37,113.97,False
4,AAPL,213.50,626.05,False
5,MSFT,192.97,354.90,False
6,NVDA,959.85,3822.84,False
7,GOOGL,245.40,535.47,False
8,AMZN,81.72,278.79,False
9,META,275.28,204.42,True


In [18]:
weekly_signals = []

for ticker, df in results.items():
    latest = df.iloc[-1]
    weekly_signals.append({
        "Ticker": ticker,
        "Signal": latest["signal"],
        "Price": latest["Close"],
        "RSI": latest["rsi"]
    })

signals_df = pd.DataFrame(weekly_signals)
signals_df.sort_values("Signal")


,Ticker,Signal,Price,RSI
0,SPY,BUY,744.39,66.06
22,ABBV,BUY,230.01,58.73
23,MRK,BUY,115.48,54.75
48,XLF,BUY,53.70,59.10
25,GE,BUY,355.12,64.52
27,DE,BUY,598.59,58.84
28,HON,BUY,228.11,54.48
33,SBUX,BUY,100.15,53.94
21,JNJ,BUY,231.29,54.71
34,XOM,BUY,138.47,47.19
